# 나란히SDK 보행 장애물 탐지 모델 — 학습부터 배포 파일까지

AI Hub 「인도보행 영상」으로 YOLOv8n을 파인튜닝하고, 단말에서 돌릴 int8 ONNX까지
만드는 과정 전부입니다. 위에서 아래로 한 번 읽으면 흐름이 드러납니다.

```
CVAT XML + 이미지          AI Hub 원본
      │
      │  1. 데이터 준비 — YOLO 형식으로 변환, 클래스 29종을 24종으로 병합
      ▼
  YOLO 데이터셋            images/ labels/ street.yaml
      │
      │  2. 학습 — YOLOv8n 에서 100 epochs
      ▼
     best.pt                6.2 MB
      │
      │  3. ONNX 변환
      ▼
    float32.onnx           12.3 MB
      │
      │  4. int8 양자화
      ▼
    model.onnx             3.5 MB   ← 배포하는 파일
      │
      │  5. 추론 확인
      ▼
   탐지 결과
```

**AI Hub 원본 이미지와 라벨은 이 저장소에 없습니다.** 데이터 이용 조건상 배포할 수
없어 변환하는 코드만 두었습니다.

## 1. 데이터 준비

AI Hub가 주는 어노테이션은 **CVAT XML**입니다. Ultralytics가 읽는 **YOLO 형식**으로
바꿉니다. 이미지 한 장에 텍스트 파일 하나이고, 한 줄이 박스 하나입니다.

```
12 0.481250 0.627431 0.092188 0.351042
클래스  중심x     중심y     너비     높이     ← 0~1 로 정규화
```

여기서 함께 하는 일이 셋입니다.

**클래스를 29종에서 24종으로 합칩니다.** 표본이 부족하거나 형태가 겹쳐 구분이 실익이
없는 것들입니다.

| 합친 뒤 | 원래 |
|---|---|
| `micromobility` | bicycle, scooter |
| `push_cart` | carrier, stroller |
| `pet` | cat, dog |
| `service_terminal` | kiosk, parking_meter |
| `control_cabinet` | power_controller, traffic_light_controller |

**클래스 이름을 COCO 표기에 맞춥니다.** `fire_hydrant`를 `fire hydrant`로 바꾸는 식입니다.
COCO 사전 학습 가중치에서 이어서 학습하므로 이름이 같으면 대응이 분명해집니다.

**학습과 검증을 영상 묶음 단위로 나눕니다.** 이것이 가장 중요합니다. 데이터가 영상에서
뽑은 연속 프레임이라 **장 단위로 무작위 분리하면 거의 같은 장면이 양쪽에 들어가** 검증
점수가 부풀려집니다. 폴더 경로의 SHA-256 해시로 배정해 실행할 때마다 같은 결과가
나오게 했습니다.

In [ ]:
from pathlib import Path
import hashlib
import os
import xml.etree.ElementTree as ET
from collections import Counter

# AI Hub 「인도보행 영상」의 바운딩박스 폴더 (Bbox_1_new ... Bbox_30_new 가 들어 있는 곳)
SOURCE_ROOT = Path(r"G:\dataset\인도보행 학습자료(뎁스프리딕션,바운딩박스)\인도보행 영상\바운딩박스")
OUTPUT_DIR = Path(r"G:\ai_ws\yolo_test\4.yolo_all_data_train_test\dataset_full_merged")

TRAIN_RATIO = 0.8
PAD_VALUE = 114

# 클래스 24종. 순서가 곧 클래스 번호이므로 바꾸면 안 된다.
CLASS_NAMES = [
    "micromobility",      # 0  bicycle + scooter
    "bus",                # 1
    "car",                # 2
    "push_cart",          # 3  carrier + stroller
    "pet",                # 4  cat + dog
    "motorcycle",         # 5
    "movable_signage",    # 6
    "person",             # 7
    "truck",              # 8
    "wheelchair",         # 9
    "barricade",          # 10
    "bench",              # 11
    "bollard",            # 12
    "chair",              # 13
    "fire hydrant",       # 14
    "service_terminal",   # 15  kiosk + parking_meter
    "pole",               # 16
    "potted plant",       # 17
    "control_cabinet",    # 18  power_controller + traffic_light_controller
    "transit_stop",       # 19
    "table",              # 20
    "traffic light",      # 21
    "traffic_sign",       # 22
    "tree_trunk",         # 23
]

# XML 의 클래스 이름을 위 목록의 이름으로 옮긴다.
REMAP = {
    # COCO 표기에 맞춘다
    "fire_hydrant": "fire hydrant",
    "potted_plant": "potted plant",
    "traffic_light": "traffic light",
    # 병합
    "bicycle": "micromobility",
    "scooter": "micromobility",
    "carrier": "push_cart",
    "stroller": "push_cart",
    "cat": "pet",
    "dog": "pet",
    "kiosk": "service_terminal",
    "parking_meter": "service_terminal",
    "power_controller": "control_cabinet",
    "traffic_light_controller": "control_cabinet",
    "stop": "transit_stop",
}

CLASS_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}

In [ ]:
def 배정(묶음경로: Path) -> str:
    """폴더 묶음 하나를 통째로 train 또는 val 에 넣는다.

    파이썬 hash() 는 실행마다 달라지므로 SHA-256 을 쓴다. 그래야 다시 돌려도
    같은 폴더가 같은 쪽으로 간다.
    """
    값 = int(hashlib.sha256(묶음경로.as_posix().lower().encode()).hexdigest()[:8], 16)
    return "train" if 값 / 0xFFFFFFFF < TRAIN_RATIO else "val"


def 박스변환(box, 너비: int, 높이: int):
    """CVAT 의 모서리 좌표를 YOLO 의 정규화된 중심과 크기로 바꾼다."""
    def 자르기(v, 최대):
        return max(0.0, min(float(v), 최대))

    x1 = 자르기(box.attrib["xtl"], 너비)
    y1 = 자르기(box.attrib["ytl"], 높이)
    x2 = 자르기(box.attrib["xbr"], 너비)
    y2 = 자르기(box.attrib["ybr"], 높이)

    # 이미지 밖으로 나간 박스를 잘라내면 넓이가 0 이 되는 것이 나온다. 버린다.
    if x2 - x1 <= 0 or y2 - y1 <= 0:
        return None

    return ((x1 + x2) / 2 / 너비, (y1 + y2) / 2 / 높이,
            (x2 - x1) / 너비, (y2 - y1) / 높이)


def 라벨쓰기(낼곳: Path, 이미지요소, 셈: Counter) -> int:
    너비 = int(이미지요소.attrib["width"])
    높이 = int(이미지요소.attrib["height"])
    줄들 = []

    for box in 이미지요소.findall("box"):
        이름 = REMAP.get(box.attrib["label"], box.attrib["label"])
        if 이름 not in CLASS_TO_ID:
            raise RuntimeError(f"목록에 없는 클래스: {box.attrib['label']} -> {이름}")

        값 = 박스변환(box, 너비, 높이)
        if 값 is None:
            continue

        cx, cy, w, h = 값
        줄들.append(f"{CLASS_TO_ID[이름]} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
        셈[이름] += 1

    낼곳.parent.mkdir(parents=True, exist_ok=True)
    낼곳.write_text("\n".join(줄들) + ("\n" if 줄들 else ""), encoding="utf-8")
    return len(줄들)

In [ ]:
def 변환하기():
    # **출력 폴더가 비어 있지 않으면 멈춘다.** 클래스 목록이나 TRAIN_RATIO 를 바꾼 뒤
    # 같은 폴더에 다시 돌리면, 라벨은 덮어써지지만 옛 이미지 링크는 남는다. 배정이
    # 달라지면 같은 이미지가 train 과 val 양쪽에 남아 **누수 방지가 무너진다.**
    if OUTPUT_DIR.exists() and any(p.is_file() for p in OUTPUT_DIR.rglob("*")):
        raise RuntimeError(
            f"출력 폴더가 비어 있지 않다: {OUTPUT_DIR}\n"
            "옛 결과와 섞이지 않도록 새 폴더 이름을 쓰거나 먼저 비워라.")

    xml들 = sorted(p for p in SOURCE_ROOT.rglob("*.xml")
                   if any(부분.lower().startswith("bbox_") for 부분 in p.parts))
    if not xml들:
        raise RuntimeError(f"XML 을 찾지 못했다: {SOURCE_ROOT}")
    print("XML", len(xml들), "개")

    통계 = {쪽: {"묶음": set(), "이미지": 0, "박스": 0, "클래스": Counter()}
            for 쪽 in ("train", "val")}

    for n, xml in enumerate(xml들, 1):
        묶음 = xml.parent.relative_to(SOURCE_ROOT)
        쪽 = 배정(묶음)
        통계[쪽]["묶음"].add(묶음.as_posix())

        for 요소 in ET.parse(xml).getroot().findall("image"):
            파일명 = Path(요소.attrib["name"]).name
            원본 = xml.parent / 파일명
            if not 원본.exists():
                continue

            # **이미지는 복사하지 않고 하드 링크로 건다.** 35만 장을 복사하면
            # 디스크가 두 배로 든다. 원본과 출력이 같은 NTFS 드라이브여야 한다.
            사본 = OUTPUT_DIR / "images" / 쪽 / 묶음 / 파일명
            사본.parent.mkdir(parents=True, exist_ok=True)
            if not 사본.exists():
                os.link(원본, 사본)

            박스수 = 라벨쓰기(
                OUTPUT_DIR / "labels" / 쪽 / 묶음 / f"{Path(파일명).stem}.txt",
                요소, 통계[쪽]["클래스"])

            통계[쪽]["이미지"] += 1
            통계[쪽]["박스"] += 박스수

        if n % 100 == 0 or n == len(xml들):
            print(f"  {n}/{len(xml들)}")

    # Ultralytics 가 읽을 데이터셋 정의
    yaml = OUTPUT_DIR / "street.yaml"
    yaml.write_text("\n".join([
        f"path: {OUTPUT_DIR.as_posix()}", "",
        "train: images/train", "val: images/val", "", "names:",
        *(f"  {i}: {이름}" for i, 이름 in enumerate(CLASS_NAMES)),
    ]) + "\n", encoding="utf-8")

    for 쪽 in ("train", "val"):
        r = 통계[쪽]
        print(f"[{쪽}] 묶음 {len(r['묶음'])}  이미지 {r['이미지']}  박스 {r['박스']}")

    return yaml


DATA_YAML = 변환하기()

## 2. 학습

**COCO 사전 학습 가중치(`yolov8n.pt`)에서 이어서 학습합니다.** 처음부터 학습하지
않습니다. `person`, `car`처럼 COCO 에도 있는 클래스는 이미 배운 것을 물려받고,
`bollard`, `tree_trunk`처럼 없는 클래스만 새로 배우면 되기 때문입니다.

`optimizer="auto"` 이므로 Ultralytics 가 데이터셋 크기를 보고 직접 고릅니다. 실제 학습
로그에 이렇게 남았습니다.

```
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937'
           and determining best 'optimizer', 'lr0' and 'momentum' automatically...
optimizer: MuSGD(lr=0.01, momentum=0.9) with parameter groups
           57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
```

**아래에 적은 `lr0` 과 `momentum` 은 요청값일 뿐 실제로 쓰이지 않았습니다.** `auto` 가
무시하고 다시 정합니다. 실효값은 **`MuSGD`, `lr=0.01`, `momentum=0.9`, `weight_decay=0.0005`**
입니다. `runs/.../args.yaml` 에는 요청값이 그대로 찍히므로 그 파일만 보면 어긋납니다.

**`auto` 는 `warmup_bias_lr` 도 0 으로 바꿉니다.** 기본값은 0.1 이고 `args.yaml` 에도
0.1 로 남지만 실제로는 0 으로 돌았습니다. 옵티마이저를 명시적으로 적을 때 이것을
빠뜨리면 warmup 이 달라집니다.

`patience=10` 은 검증 점수가 10 epochs 동안 나아지지 않으면 멈춘다는 뜻입니다.
`save_period=1` 로 매 epoch 를 남겨 두면 중간에 끊겨도 이어갈 수 있습니다.

RTX 5080 (16GB) 에서 `batch=32` 로 돌렸습니다.

In [ ]:
from ultralytics import YOLO

PROJECT = Path(r"G:\ai_ws\yolo_test\4.yolo_all_data_train_test\runs\full")
RUN_NAME = "yolov8n_street_class_merged_1"

model = YOLO("yolov8n.pt")

results = model.train(
    data=str(DATA_YAML),
    epochs=100,
    patience=10,
    imgsz=640,
    batch=32,
    device=0,
    workers=6,
    warmup_epochs=3,
    # **auto 가 아래 lr0 과 momentum 을 무시하고 다시 정한다.**
    # 이 데이터에서 실제로 쓰인 값은 이렇다. 명시적으로 적으려면 아래 넷을 그대로 준다.
    #
    #     optimizer="MuSGD",     # auto 가 고른 것 (iteration 이 1만을 넘으면 MuSGD)
    #     lr0=0.01,
    #     momentum=0.9,          # 기본값 0.937 이 아니다
    #     warmup_bias_lr=0.0,    # auto 가 조용히 0 으로 바꾼다. 기본값은 0.1 이다
    #
    optimizer="auto",
    seed=0,
    deterministic=True,
    amp=True,
    max_det=100,
    project=str(PROJECT),
    name=RUN_NAME,
    save=True,
    save_period=1,
    plots=True,
)

# **이름을 조립하지 말고 실제 저장 위치를 받는다.** 같은 이름의 폴더가 있으면
# Ultralytics 가 뒤에 번호를 붙여 `..._12` 로 저장한다. 조립한 경로는 그것을 못 따라가
# **두 번째 실행부터 낡은 가중치를 조용히 내보내게 된다.**
BEST_PT = Path(results.save_dir) / "weights" / "best.pt"
print("학습 완료:", BEST_PT, f"{BEST_PT.stat().st_size / 1e6:.1f} MB")

## 3. ONNX 변환

브라우저와 모바일에서 돌리려면 ONNX 여야 합니다.

**`nms=False` 입니다.** NMS 를 모델 안에 넣지 않고 밖에서 합니다. 그래야 신뢰도와 IoU
문턱값을 실행할 때 바꿀 수 있고, 웹과 모바일 구현이 같은 후처리를 공유합니다.
대신 출력이 날것이라 5 단계의 후처리가 반드시 필요합니다.

`dynamic=False` 로 입력을 640×640 에 고정합니다. 크기가 고정되어야 양자화가 안정적이고
모바일 런타임에서도 빠릅니다.

In [ ]:
from ultralytics import YOLO

FLOAT32_ONNX = YOLO(str(BEST_PT)).export(
    format="onnx",
    imgsz=640,
    batch=1,
    dynamic=False,
    simplify=True,
    opset=17,
    nms=False,      # 후처리는 밖에서 한다
)
print("ONNX:", FLOAT32_ONNX)

## 4. int8 양자화

12.3 MB 를 **3.5 MB** 로 줄입니다. mAP50 은 1.4% 만 떨어집니다.

가중치를 8비트로 줄이려면 각 층의 값이 어느 범위에 놓이는지 알아야 합니다. 그래서
**실제 이미지 몇 장을 흘려보내 범위를 재는데(calibration)**, 여기서 세 가지가 어긋나기
쉽습니다.

**calibration 이미지는 학습 세트에서 뽑습니다.** 검증 세트로 하면 그 이미지에 맞춰진
범위가 잡혀 mAP 가 부풀려집니다. 실측으로 1.2% 차이가 났습니다.

**한쪽에 치우치지 않게 목록에서 고르게 건너뛰며 뽑습니다.** 밝은 사진만 보면 어두운
장면의 값 범위를 놓칠까 걱정해 넣은 장치인데, **실제로 재보니 거의 차이가 없었습니다** —
밝기 구간별 mAP50 손실이 가장 어두운 구간 -2.1%, 가장 밝은 구간 -2.1% 로 같았습니다.
아래에서 쓰는 percentile calibration 이 이미 그 일을 하고 있는 것으로 보입니다.

**detection head 에서 Conv 가 아닌 노드는 양자화에서 뺍니다.** 이걸 빼지 않으면
**클래스 점수가 통째로 0 이 됩니다.** 출력 텐서 하나에 박스(0~640)와 클래스 점수(0~1)가
같이 들어 있어서, 범위가 박스 쪽으로 잡히면 점수가 전부 뭉개지기 때문입니다.
head 는 언제나 마지막 `model.N` 블록이므로 번호를 직접 적지 말고 찾아서 씁니다.

In [ ]:
import re
import numpy as np
import onnx
import onnxruntime as ort
from onnxruntime.quantization import (
    CalibrationDataReader, CalibrationMethod, QuantFormat, QuantType, quantize_static,
)
from onnxruntime.quantization.shape_inference import quant_pre_process
import cv2

INT8_ONNX = Path("model.onnx")
CALIB_ROOT = OUTPUT_DIR / "images" / "train"
MAX_CALIB = 128     # percentile 은 중간 텐서를 다 들고 있어 많으면 메모리에서 터진다


def 전처리(경로: Path) -> np.ndarray:
    """추론과 똑같은 레터박스. 5 단계와 같은 규칙이어야 한다."""
    img = cv2.cvtColor(cv2.imread(str(경로)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    배율 = min(640 / h, 640 / w)
    nw, nh = round(w * 배율), round(h * 배율)
    좌 = round((640 - nw) / 2 - 0.1)
    상 = round((640 - nh) / 2 - 0.1)

    판 = np.full((640, 640, 3), PAD_VALUE, np.uint8)
    판[상:상 + nh, 좌:좌 + nw] = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)
    return (판.transpose(2, 0, 1)[None].astype(np.float32) / 255.0)


# 한쪽에 치우치지 않도록 목록에서 고르게 건너뛰며 뽑는다
전부 = sorted(CALIB_ROOT.rglob("*.jpg"))
if not 전부:
    raise RuntimeError(f"calibration 이미지가 없다: {CALIB_ROOT}")
걸음 = max(1, len(전부) // MAX_CALIB)
사진들 = 전부[::걸음][:MAX_CALIB]
print(f"calibration {len(전부)}장 중 {len(사진들)}장")

# shape 추론과 상수 폴딩을 먼저 해 둬야 양자화가 안정적이다
준비 = Path("prepared.onnx")
quant_pre_process(str(FLOAT32_ONNX), str(준비), skip_symbolic_shape=False)

# detection head 는 마지막 model.N 블록이다. 번호를 적어 두면 다른 모델에서 깨진다.
m = onnx.load(str(준비))
층 = {int(g.group(1)) for n in m.graph.node
      if (g := re.match(r"^/model\.(\d+)/", n.name or ""))}
if not 층:
    raise RuntimeError(
        "/model.N/ 이름을 가진 노드가 없다. 이 코드는 Ultralytics 가 내보낸 ONNX 를"
        " 전제한다. 다른 도구로 내보낸 것이면 head 를 직접 찾아야 한다.")
머리 = max(층)
제외 = [n.name for n in m.graph.node
        if n.name.startswith(f"/model.{머리}/") and n.op_type != "Conv"]
print(f"detection head = /model.{머리}/ , 양자화에서 뺄 노드 {len(제외)}개")

입력이름 = onnx.load(str(FLOAT32_ONNX)).graph.input[0].name


class 읽개(CalibrationDataReader):
    """이미지를 한 장씩 흘려보낸다. 128장을 한 배열에 담으면 메모리에서 터진다."""

    def __init__(self):
        self.i = 0

    def get_next(self):
        if self.i >= len(사진들):
            return None
        x = 전처리(사진들[self.i])
        self.i += 1
        return {입력이름: x}

    def rewind(self):
        self.i = 0


quantize_static(
    model_input=str(준비),
    model_output=str(INT8_ONNX),
    calibration_data_reader=읽개(),
    quant_format=QuantFormat.QDQ,
    activation_type=QuantType.QUInt8,
    weight_type=QuantType.QInt8,
    per_channel=True,
    reduce_range=False,
    calibrate_method=CalibrationMethod.Percentile,
    nodes_to_exclude=제외,
    extra_options={"CalibMovingAverage": False, "percentile": 99.999},
)
준비.unlink(missing_ok=True)

print(f"{INT8_ONNX} {INT8_ONNX.stat().st_size / 1e6:.1f} MB")

# **클래스 점수가 살아 있는지 반드시 본다.** 최대가 0 이면 양자화가 망가진 것이다.
for 라벨, 경로 in [("float32", FLOAT32_ONNX), ("int8", INT8_ONNX)]:
    s = ort.InferenceSession(str(경로), providers=["CPUExecutionProvider"])
    (o,) = s.run(None, {s.get_inputs()[0].name: 전처리(사진들[0])})
    최고 = o[0][4:, :].max(axis=0)
    print(f"  {라벨:<8} 클래스 점수 최대 {최고.max():.5f}   0.25 넘는 후보 {(최고 > 0.25).sum()}")

## 5. 추론 확인

만들어진 `model.onnx` 가 실제로 도는지 봅니다. **여기 있는 전처리와 후처리가 곧
배포되는 SDK 가 하는 일**이라, 이 결과가 맞으면 변환이 제대로 된 것입니다.

출력 `[1, 28, 8400]` 은 후보 8400 개이고, 각 후보가 값 28 개를 가집니다.

```
[0:4]   박스 — 중심x, 중심y, 너비, 높이   (640 좌표계)
[4:28]  클래스 24 종의 점수
```

**클래스 이름을 코드에 적지 않고 모델에서 읽습니다.** 모델을 다시 학습해 클래스가
바뀌어도 이 코드는 그대로 돕니다.

후처리에서 세 가지를 맞춰야 배포 구현과 같은 답이 나옵니다.

**IoU 문턱값은 0.7 입니다.** 흔히 쓰는 0.45 로 하면 겹쳐 선 차량 중 일부가 지워집니다.

**점수를 정렬할 때 안정 정렬을 씁니다.** int8 양자화가 점수를 이산값으로 만들어
**동점이 아주 많이 생깁니다** — 한 사진에서 문턱을 넘은 후보 68 개 중 서로 다른 점수가
23 개뿐이었습니다. NMS 는 먼저 집힌 쪽이 살아남는 구조라 정렬이 흔들리면 결과가 바뀝니다.

**화면 밖으로 나간 박스를 자르는 것은 NMS 를 건 뒤에 합니다.** 크기를 되돌리고 옮기는
것은 IoU 를 그대로 두지만 **자르는 것은 넓이와 교집합을 바꿉니다.** 먼저 자르면 가장자리에
걸친 박스의 억제 결과가 달라집니다.

In [ ]:
import ast
import numpy as np
import onnxruntime as ort
import cv2

CONF, IOU = 0.25, 0.7
IMAGE = Path(r"확인할_사진.jpg")

sess = ort.InferenceSession(str(INT8_ONNX), providers=["CPUExecutionProvider"])
이름들 = ast.literal_eval(sess.get_modelmeta().custom_metadata_map["names"])

# ── 전처리 (4 단계와 같은 규칙) ──────────────────────────────────
if not IMAGE.exists():
    raise FileNotFoundError(f"확인할 사진이 없다: {IMAGE}")

원본 = cv2.cvtColor(cv2.imread(str(IMAGE)), cv2.COLOR_BGR2RGB)
h, w = 원본.shape[:2]
배율 = min(640 / h, 640 / w)
nw, nh = round(w * 배율), round(h * 배율)
좌 = round((640 - nw) / 2 - 0.1)
상 = round((640 - nh) / 2 - 0.1)

판 = np.full((640, 640, 3), PAD_VALUE, np.uint8)
판[상:상 + nh, 좌:좌 + nw] = cv2.resize(원본, (nw, nh), interpolation=cv2.INTER_AREA)
x = 판.transpose(2, 0, 1)[None].astype(np.float32) / 255.0

# ── 실행 ────────────────────────────────────────────────────────
(출력,) = sess.run(None, {sess.get_inputs()[0].name: x})   # (1, 28, 8400)
후보 = 출력[0].T                                            # (8400, 28)

# ── 후처리 ──────────────────────────────────────────────────────
점수 = 후보[:, 4:].max(axis=1)
클래스 = 후보[:, 4:].argmax(axis=1)
남길 = 점수 >= CONF          # 경계값을 살린다. 동점이 많아 실제로 갈리는 자리다.
박스, 점수, 클래스 = 후보[남길, :4], 점수[남길], 클래스[남길]

# 중심과 크기를 모서리로 바꾼다. **아직 640 좌표계이고 자르지 않는다.**
모서리 = np.empty(박스.shape, np.float64)
모서리[:, 0] = 박스[:, 0] - 박스[:, 2] / 2
모서리[:, 1] = 박스[:, 1] - 박스[:, 3] / 2
모서리[:, 2] = 박스[:, 0] + 박스[:, 2] / 2
모서리[:, 3] = 박스[:, 1] + 박스[:, 3] / 2


def nms(상자: np.ndarray, 점: np.ndarray, 문턱: float) -> np.ndarray:
    """점수가 높은 것부터 남기고, 많이 겹치는 것을 지운다."""
    x1, y1, x2, y2 = 상자[:, 0], 상자[:, 1], 상자[:, 2], 상자[:, 3]
    넓이 = np.maximum(0, x2 - x1) * np.maximum(0, y2 - y1)

    # **안정 정렬이어야 한다.** 동점이 많아서 순서가 흔들리면 결과가 달라진다.
    순서 = np.argsort(-점, kind="stable")

    남김 = []
    while 순서.size:
        i = 순서[0]
        남김.append(int(i))
        if 순서.size == 1:
            break

        나머지 = 순서[1:]
        겹w = np.maximum(0, np.minimum(x2[i], x2[나머지]) - np.maximum(x1[i], x1[나머지]))
        겹h = np.maximum(0, np.minimum(y2[i], y2[나머지]) - np.maximum(y1[i], y1[나머지]))
        교집합 = 겹w * 겹h
        합집합 = 넓이[i] + 넓이[나머지] - 교집합
        순서 = 나머지[교집합 / (합집합 + 1e-9) <= 문턱]

    return np.asarray(남김, dtype=np.int64)


# **NMS 는 클래스별로 건다.** 다 같이 걸면 겹쳐 선 사람과 볼라드 중 하나가 지워진다.
살아남음 = []
for c in np.unique(클래스):
    골라 = np.flatnonzero(클래스 == c)
    for i in nms(모서리[골라], 점수[골라], IOU):
        살아남음.append((모서리[골라][i], float(점수[골라][i]), int(c)))

# **여기서 비로소 원본 좌표로 옮기고 자른다.** 자르기를 NMS 앞에 두면 억제가 달라진다.
결과 = []
for 상자, 점, c in 살아남음:
    x1 = min(max((상자[0] - 좌) / 배율, 0), w)
    y1 = min(max((상자[1] - 상) / 배율, 0), h)
    x2 = min(max((상자[2] - 좌) / 배율, 0), w)
    y2 = min(max((상자[3] - 상) / 배율, 0), h)
    결과.append((np.array([x1, y1, x2, y2]), 점, c))

print(f"탐지 {len(결과)}개\n")
for 상자, 점, c in sorted(결과, key=lambda t: -t[1]):
    x1, y1, x2, y2 = 상자.round().astype(int)
    print(f"  {이름들[c]:<18} {점:.3f}   ({x1}, {y1}) - ({x2}, {y2})")

# 그려서 저장
그림 = cv2.cvtColor(원본, cv2.COLOR_RGB2BGR).copy()
for 상자, 점, c in 결과:
    x1, y1, x2, y2 = 상자.round().astype(int)
    cv2.rectangle(그림, (x1, y1), (x2, y2), (0, 200, 0), 2)
    cv2.putText(그림, f"{이름들[c]} {점:.2f}", (x1, max(0, y1 - 6)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 200, 0), 1)
cv2.imwrite("결과.jpg", 그림)
print("\n결과.jpg 로 저장했습니다.")